# 25 · Corrective / Self-RAG

**Idea:** naive RAG trusts whatever the retriever returns. Corrective RAG (CRAG) adds a feedback loop — an LLM grades each retrieved chunk, and if too few are relevant it rewrites the query and retrieves again *before* answering. This is a lightweight cousin of Self-RAG's reflection tokens: the model critiques its own evidence and acts on it, rescuing weak first-pass retrievals.

> The grade/rewrite loop makes several calls, so it defaults to Ollama. One cell optionally uses DeepSeek — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Build a store. On a **well-phrased** query the first retrieval already passes the relevance grade, so no rewrite happens.

In [ ]:
from rag_lab import load_markdown, NumpyStore, corrective_answer
from study_buddy import CostTracker

store = NumpyStore(provider='ollama'); store.add(load_markdown())

tracker = CostTracker()
answer, trace = corrective_answer(
    'What is reranking in a retrieval pipeline?', store,
    provider='ollama', k=4, min_relevant=2, tracker=tracker,
)
print(answer)
print('\nqueries tried:', trace.queries)
print('rewrites:', trace.rewrites, '| kept chunks:', len(trace.kept))

Now a **vague** query. The grader rejects most first-pass chunks, so CRAG rewrites the query and retrieves again — watch `rewrites` tick up and a second entry appear in `queries`.

In [ ]:
answer2, trace2 = corrective_answer(
    'tell me about the search thing', store,
    provider='ollama', k=4, min_relevant=3, max_rewrites=1, tracker=tracker,
)
print(answer2)
print('\nqueries tried:', trace2.queries)
print('rewrites:', trace2.rewrites)
print('cost so far:', tracker.summary())

Optionally run the loop on DeepSeek. Skips if no key.

In [ ]:
try:
    cloud = CostTracker()
    a, t = corrective_answer('tell me about the search thing', store,
                             provider='deepseek', k=4, min_relevant=3, tracker=cloud)
    print('rewrites:', t.rewrites, '| queries:', t.queries)
    print('cost:', cloud.summary())
except Exception as e:
    print('deepseek skipped:', type(e).__name__, e)

**Takeaway:** CRAG spends extra LLM calls to *self-correct* bad retrieval instead of confidently answering from irrelevant context. The `min_relevant` threshold and `max_rewrites` budget trade cost for robustness — turn them up for high-stakes answers, down for cheap ones.

Next up → `26_security_guardrails.ipynb`, where retrieved documents become an attack surface.